<h1><center>SpaGCN Tutorial</center></h1>


<center>Author: Jian Hu,*, Xiangjie Li, Kyle Coleman, Amelia Schroeder, Nan Ma, David J. Irwin, Edward B. Lee, Russell T. Shinohara, Mingyao Li*

### Outline
1. Installation
2. Import modules
3. Read in data
4. Integrate gene expression and histology into a Graph
5. Spatial domain detection using SpaGCN
6. Identify SVGs
7. Identify Meta Gene
8. Multiple tissue sections analysis

### 1. Installation
To install SpaGCN package you must make sure that your python version is over 3.5.=. If you don’t know the version of python you can check it by:

In [5]:
import platform
platform.python_version()

'3.7.12'

Note: Because SpaGCN pends on pytorch, you should make sure torch is correctly installed.
<br>
Now you can install the current release of SpaGCN by the following three ways:
#### 1.1 PyPI: Directly install the package from PyPI.

In [ ]:
pip3 install SpaGCN
#Note: you need to make sure that the pip is for python3，or we should install SpaGCN by
python3 -m pip install SpaGCN
pip3 install SpaGCN
#If you do not have permission (when you get a permission denied error), you should install SpaGCN by
pip3 install --user SpaGCN

#### 1.2 Github
Download the package from Github and install it locally:

In [ ]:
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py install --user

#### 1.3 Anaconda
If you do not have Python3.5 or Python3.6 installed, consider installing Anaconda (see Installing Anaconda). After installing Anaconda, you can create a new environment, for example, SpaGCN (you can change to any name you like).

In [ ]:
#create an environment called SpaGCN
conda create -n SpaGCN python=3.7.9
#activate your environment 
conda activate SpaGCN
git clone https://github.com/jianhuupenn/SpaGCN
cd SpaGCN/SpaGCN_package/
python3 setup.py build
python3 setup.py install
conda deactivate

### 2. Import python modules

In [1]:
import os,csv,re
import pandas as pd
import numpy as np
import scanpy as sc
import math
import SpaGCN as spg
from scipy.sparse import issparse
import random, torch
import warnings
warnings.filterwarnings("ignore")
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import SpaGCN as spg
#In order to read in image data, we need to install some package. Here we recommend package "opencv"
#inatll opencv in python
#!pip3 install opencv-python
import cv2

In [2]:
spg.__version__

'1.2.7'

### 3. Read in data
The current version of SpaGCN requres three input data: 
<br>
1. The gene expression matrix(n by k): expression_matrix.h5;
<br>
2. Spatial coordinateds of samplespositions.txt;
<br>
3. Histology image(optional): histology.tif, can be tif or png or jepg.
<br>
The gene expreesion data can be stored as an AnnData object. AnnData stores a data matrix .X together with annotations of observations .obs, variables .var and unstructured annotations .uns. 

In [3]:
"""
#Read original 10x_h5 data and save it to h5ad
from scanpy import read_10x_h5
adata = read_10x_h5("../tutorial/data/151673/expression_matrix.h5")
spatial=pd.read_csv("../tutorial/data/151673/positions.txt",sep=",",header=None,na_filter=False,index_col=0) 
adata.obs["x1"]=spatial[1]
adata.obs["x2"]=spatial[2]
adata.obs["x3"]=spatial[3]
adata.obs["x4"]=spatial[4]
adata.obs["x5"]=spatial[5]
adata.obs["x_array"]=adata.obs["x2"]
adata.obs["y_array"]=adata.obs["x3"]
adata.obs["x_pixel"]=adata.obs["x4"]
adata.obs["y_pixel"]=adata.obs["x5"]
#Select captured samples
adata=adata[adata.obs["x1"]==1]
adata.var_names=[i.upper() for i in list(adata.var_names)]
adata.var["genename"]=adata.var.index.astype("str")
adata.write_h5ad("data/151673/sample_data.h5ad")
"""
#Read in gene expression and spatial location
adata=sc.read("data/151673/sample_data.h5ad")
#Read in hitology image
img=cv2.imread("data/151673/histology.tif")

### 4. Integrate gene expression and histology into a Graph

In [4]:
#Set coordinates
x_array=adata.obs["x1"].tolist()
y_array=adata.obs["x2"].tolist()
x_pixel=adata.obs["x3"].tolist()
y_pixel=adata.obs["x4"].tolist()

#Test coordinates on the image
img_new=img.copy()
for i in range(len(x_pixel)):
    x=x_pixel[i]
    y=y_pixel[i]
    img_new[int(x-20):int(x+20), int(y-20):int(y+20),:]=0

cv2.imwrite('./sample_results/151673_map.jpg', img_new)

True

In [5]:
print(adata.obs.columns)

Index(['x1', 'x2', 'x3', 'x4', 'x5'], dtype='object')


- The ‘s’ parameter determines the weight given to histology when calculating Euclidean distance between every two spots. ‘s = 1’ means that the histology pixel intensity value has the same scale variance as the (x,y) coordinates, whereas higher value of ‘s’ indicates higher scale variance, hence, higher weight to histology, when calculating the Euclidean distance. 

- The "b"parameter determines the area of each spot when extracting color intensity.

In [6]:
#Calculate adjacent matrix
s=1
b=49
adj=spg.calculate_adj_matrix(x=x_pixel,y=y_pixel, x_pixel=x_pixel, y_pixel=y_pixel, image=img, beta=b, alpha=s, histology=True)
#If histlogy image is not available, SpaGCN can calculate the adjacent matrix using the fnction below
#adj=calculate_adj_matrix(x=x_pixel,y=y_pixel, histology=False)
np.savetxt('./data/151673/adj.csv', adj, delimiter=',')

Calculateing adj matrix using histology image...
Var of c0,c1,c2 =  2.656835035703222 4.591315698217075 3.3332271661282666
Var of x,y,z =  942.1320230796399 5606737.526317932 5606737.526317933


### 5. Spatial domain detection using SpaGCN

#### 5.1 Expression data preprocessing

In [7]:
adata=sc.read("./data/151673/sample_data.h5ad")
adj=np.loadtxt('./data/151673/adj.csv', delimiter=',')
adata.var_names_make_unique()
spg.prefilter_genes(adata,min_cells=3) # avoiding all genes are zeros
spg.prefilter_specialgenes(adata)
#Normalize and take log for UMI
sc.pp.normalize_per_cell(adata)
sc.pp.log1p(adata)

#### 5.2 Set hyper-parameters

- p: Percentage of total expression contributed by neighborhoods.
- l: Parameter to control p.

In [8]:
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

Run 1: l [0.01, 1000], p [0.0, 640.7861957803858]
Run 2: l [0.01, 500.005], p [0.0, 241.2276308648813]
Run 3: l [0.01, 250.0075], p [0.0, 86.65013073208404]
Run 4: l [0.01, 125.00874999999999], p [0.0, 30.92014710823589]
Run 5: l [0.01, 62.509375], p [0.0, 11.890337066262287]
Run 6: l [0.01, 31.2596875], p [0.0, 5.212647230462018]
Run 7: l [0.01, 15.63484375], p [0.0, 2.206147288757988]
Run 8: l [0.01, 7.822421875], p [0.0, 0.8265454036986504]
Run 9: l [3.9162109375, 7.822421875], p [0.25854610679077905, 0.8265454036986504]
Run 10: l [3.9162109375, 5.86931640625], p [0.25854610679077905, 0.5257273139639844]
Run 11: l [4.8927636718750005, 5.86931640625], p [0.3867125076491458, 0.5257273139639844]
Run 12: l [5.3810400390625, 5.86931640625], p [0.4550654706616375, 0.5257273139639844]
recommended l =  5.625178222656251


- n_clusters: Number of spatial domains wanted.
- res: Resolution in the initial Louvain's Clustering methods. If the number of clusters is known, we can use the spg.search_res() fnction to search for suitable resolution(optional)

In [9]:
#If the number of clusters known, we can use the spg.search_res() fnction to search for suitable resolution(optional)
#For this toy data, we set the number of clusters=7 since this tissue has 7 layers
n_clusters=7
#Set seed
r_seed=t_seed=n_seed=100
#Seaech for suitable resolution
res=spg.search_res(adata, adj, l, n_clusters, start=0.7, step=0.1, tol=5e-3, lr=0.05, max_epochs=20, r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

Start at res =  0.7 step =  0.1
Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Res =  0.7 Num of clusters =  7
recommended res =  0.7


#### 5.3 Run SpaGCN

In [11]:
clf=spg.SpaGCN()
clf.set_l(l)
#Set seed
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
#Run
clf.train(adata,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata.obs["pred"]= y_pred
adata.obs["pred"]=adata.obs["pred"].astype('category')
#Do cluster refinement(optional)
#shape="hexagon" for Visium data, "square" for ST data.
adj_2d=spg.calculate_adj_matrix(x=x_array,y=y_array, histology=False)
refined_pred=spg.refine(sample_id=adata.obs.index.tolist(), pred=adata.obs["pred"], dis=adj_2d, shape="hexagon")
adata.obs["refined_pred"]=refined_pred
adata.obs["refined_pred"]=adata.obs["refined_pred"].astype('category')
#Save results
adata.write_h5ad("./sample_results/results.h5ad")

Initializing cluster centers with louvain, resolution =  0.7
Epoch  0
Epoch  10
Epoch  20
Epoch  30
delta_label  0.0032976092333058533 < tol  0.005
Reach tolerance threshold. Stopping training.
Total epoch: 37
Calculateing adj matrix using xy only...


In [13]:
adata = sc.read("./sample_results/results.h5ad")
print(adata.obs.columns)

Index(['x1', 'x2', 'x3', 'x4', 'x5', 'n_counts', 'pred', 'refined_pred'], dtype='object')


#### 5.4 Plot spatial domains

In [17]:
adata=sc.read("./sample_results/results.h5ad")
# Map the generic columns to the pixel names Scanpy expects
adata.obs["x_pixel"] = adata.obs["x3"]
adata.obs["y_pixel"] = adata.obs["x4"]
#Set colors used
plot_color=["#F56867","#FEB915","#C798EE","#59BE86","#7495D3","#D1D1D1","#6D1A9C","#15821E","#3A84E6","#997273","#787878","#DB4C6C","#9E7A7A","#554236","#AF5F3C","#93796C","#F9BD3F","#DAB370","#877F6C","#268785"]
#Plot spatial domains
domains="pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/pred.png", dpi=600)
plt.close()

#Plot refined spatial domains
domains="refined_pred"
num_celltype=len(adata.obs[domains].unique())
adata.uns[domains+"_colors"]=list(plot_color[:num_celltype])
ax=sc.pl.scatter(adata,alpha=1,x="y_pixel",y="x_pixel",color=domains,title=domains,color_map=plot_color,show=False,size=100000/adata.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/refined_pred.png", dpi=600)
plt.close()

**Spatial Domains**![](./sample_results/pred.png) **Refined Spatial Domains**![](./sample_results/refined_pred.png)

### 6. Identify SVGs

In [18]:
#Read in raw data
raw=sc.read("../tutorial/data/151673/sample_data.h5ad")
raw.var_names_make_unique()
raw.obs["pred"]=adata.obs["pred"].astype('category')
raw.obs["x_array"]=raw.obs["x2"]
raw.obs["y_array"]=raw.obs["x3"]
raw.obs["x_pixel"]=raw.obs["x4"]
raw.obs["y_pixel"]=raw.obs["x5"]
#Convert sparse matrix to non-sparse
raw.X=(raw.X.A if issparse(raw.X) else raw.X)
raw.raw=raw
sc.pp.log1p(raw)

- target: Target domain to identify SVGs.
- min_in_group_fraction: Minium in-group expression fraction.
- min_in_out_group_ratio: Miniumn (in-group expression fraction) / (out-group expression fraction).
- min_fold_change: Miniumn (in-group expression) / (out-group expression).
- r: Radius to detect a spot's neighboring spots.

In [21]:
print(r)

None


In [24]:
from scipy.spatial.distance import cdist

def get_adaptive_neighbor_domains(adata, target_cluster, x_col="x_pixel", y_col="y_pixel", pred_col="pred", max_neighbors=3):
    """
    Automatically calculates the optimal spatial search radius based on true data spacing
    and finds adjacent neighbor domains for a target cluster using SpaGCN.
    """
    # 1. Extract coordinates as a matrix
    coords = np.column_stack((adata.obs[x_col], adata.obs[y_col]))

    # 2. Compute distances between all points
    dist_matrix = cdist(coords, coords)

    # 3. Find the distance to the single closest neighbor for each spot (ignoring self at index 0)
    closest_neighbor_distances = np.sort(dist_matrix, axis=1)[:, 1]

    # 4. Set the radius to 1.5x the median step distance between spots
    adaptive_radius = float(np.median(closest_neighbor_distances) * 1.5)
    print(f"🤖 Dynamically calculated optimal radius: {adaptive_radius:.2f} pixels.")

    # 5. Run SpaGCN neighbor detection using the calculated radius
    neighbor_domains = spg.find_neighbor_clusters(
        target_cluster=target_cluster,
        cell_id=adata.obs.index.tolist(), 
        x=adata.obs[x_col].tolist(),  
        y=adata.obs[y_col].tolist(),
        pred=adata.obs[pred_col].tolist(),
        radius=adaptive_radius,
        ratio=1/2
    )
    
    # Return the top requested neighbor domains
    return neighbor_domains[:max_neighbors]

In [25]:
neighbors = get_adaptive_neighbor_domains(adata, target_cluster=3)
print("Neighbor domains identified:", neighbors)

🤖 Dynamically calculated optimal radius: 3.35 pixels.
radius= 3.3541019662496847 average number of neighbors for each spot is 2.9338235294117645
 Cluster 3 has neighbors:
Dmain  6 :  123
Neighbor domains identified: [6]


In [28]:
# 1. Choose your target domain (Cluster 0)
target = 0

# 2. Set filtering criteria for Spatially Variable Genes (SVGs)
min_in_group_fraction = 0.8
min_in_out_group_ratio = 1
min_fold_change = 1.5

# 3. Handle data/coordinate space and adjust radius multiplier for grid space
# Because your spots are 1-3 units apart, we increase max_neighbors lookup slightly
nbr_domians = get_adaptive_neighbor_domains(
    adata, 
    target_cluster=target, 
    x_col="x_pixel", 
    y_col="y_pixel", 
    pred_col="pred", 
    max_neighbors=3
)

# ⚠️ IF IT STILL SAYS "No neighbor domain found", force a slightly wider ring manually:
if len(nbr_domians) <= 1:
    print("🔄 Retrying neighbor lookup with a relaxed grid window...")
    # Temporarily calculate a slightly larger radius for the lookup
    coords = np.column_stack((adata.obs["x_pixel"], adata.obs["y_pixel"]))
    closest_dist = np.sort(cdist(coords, coords), axis=1)[:, 1]
    fallback_r = float(np.median(closest_dist) * 2.5) # Expanded step multiplier
    
    nbr_domians = spg.find_neighbor_clusters(
        target_cluster=target,
        cell_id=adata.obs.index.tolist(), 
        x=adata.obs["x_pixel"].tolist(),  
        y=adata.obs["y_pixel"].tolist(),
        pred=adata.obs["pred"].tolist(),
        radius=fallback_r,
        ratio=1/2
    )[:3]

# 4. FIX THE SCANPY CRASH: Inject missing metadata required by newer Scanpy versions
if 'log1p' not in adata.uns:
    adata.uns['log1p'] = {'base': None}
elif 'base' not in adata.uns['log1p']:
    adata.uns['log1p']['base'] = None

# 5. Rank genes comparing Target Domain vs. its physical neighbors
de_genes_info = spg.rank_genes_groups(
    input_adata=adata,               
    target_cluster=target,
    nbr_list=nbr_domians, 
    label_col="pred", 
    adj_nbr=True, 
    log=True
)

# 6. Filter the detected genes based on your statistical criteria
de_genes_info = de_genes_info[(de_genes_info["pvals_adj"] < 0.05)]
filtered_info = de_genes_info

filtered_info = filtered_info[
    (filtered_info["pvals_adj"] < 0.05) &
    (filtered_info["in_out_group_ratio"] > min_in_out_group_ratio) &
    (filtered_info["in_group_fraction"] > min_in_group_fraction) &
    (filtered_info["fold_change"] > min_fold_change)
]

# Sort by presence inside the target cluster
filtered_info = filtered_info.sort_values(by="in_group_fraction", ascending=False)
filtered_info["target_dmain"] = target
filtered_info["neighbors"] = str(nbr_domians)

# 7. Print the final discovered Spatially Variable Genes
print(f"\n🎉 Done! Confirmed neighbor domains used for contrast: {nbr_domians}")
print(f"SVGs for domain {target}: {filtered_info['genes'].tolist()}")

🤖 Dynamically calculated optimal radius: 3.35 pixels.
radius= 3.3541019662496847 average number of neighbors for each spot is 2.927950310559006
 Cluster 0 has neighbors:
No neighbor domain found, only return one potential neighbor domain: [2]
Try bigger radius or smaller ratio.
🔄 Retrying neighbor lookup with a relaxed grid window...
radius= 5.5901699437494745 average number of neighbors for each spot is 4.818633540372671
 Cluster 0 has neighbors:
Dmain  2 :  609
Dmain  5 :  278
Dmain  4 :  237

🎉 Done! Confirmed neighbor domains used for contrast: [2, 5, 4]
SVGs for domain 0: ['MBP']


In [29]:
filtered_info

,genes,in_group_fraction,out_group_fraction,in_out_group_ratio,in_group_mean_exp,out_group_mean_exp,fold_change,pvals_adj,target_dmain,neighbors
0,MBP,0.996273,0.964608,1.032827,2.323441,1.911369,1.509944,5.970460e-09,0,"[2, 5, 4]"


In [31]:
#Plot refinedspatial domains
color_self = clr.LinearSegmentedColormap.from_list('pink_green', ['#3AB370',"#EAE7CC","#FD1593"], N=256)
for g in filtered_info["genes"].tolist():
    raw.obs["exp"]=raw.X[:,raw.var.index==g]
    ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=g,color_map=color_self,show=False,size=100000/raw.shape[0])
    ax.set_aspect('equal', 'box')
    ax.axes.invert_yaxis()
    plt.savefig("./sample_results/"+g+".png", dpi=600)
    plt.close()


**CAMK2N1**![](./sample_results/CAMK2N1.png) **ENC1**![](./sample_results/ENC1.png) **GPM6A**![](./sample_results/GPM6A.png) **ARPP19**![](./sample_results/ARPP19.png) **HPCAL1**![](./sample_results/HPCAL1.png)

### 7. Identify Meta Gene

In [32]:
#Use domain 2 as an example
target=2
meta_name, meta_exp=spg.find_meta_gene(input_adata=raw,
                    pred=raw.obs["pred"].tolist(),
                    target_domain=target,
                    start_gene="GFAP",
                    mean_diff=0,
                    early_stop=True,
                    max_iter=3,
                    use_raw=False)

raw.obs["meta"]=meta_exp

Add gene:  TMSB10
Minus gene:  GFAP
Absolute mean change: 0.7952022552490234
Number of non-target spots reduced to: 1822
Meta gene is:  GFAP+TMSB10-GFAP
Add gene:  PCP4L1
Minus gene:  MT-ND4
Stopped! Previous Number of non-target spots 1822 359 0.7952022552490234 0.7531850337982178
Previous Number of non-target spots 1822 359 0.7952022552490234 0.7531850337982178
Previous Number of non-target spots 1822
Current Number of non-target spots 359
Absolute mean change 0.7952022552490234
Meta gene:  GFAP+TMSB10-GFAP


In [34]:
#Plot meta gene
g="GFAP"
raw.obs["exp"]=raw.X[:,raw.var.index==g]
ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=g,color_map=color_self,show=False,size=100000/raw.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/"+g+".png", dpi=600)
plt.close()

raw.obs["exp"]=raw.obs["meta"]
ax=sc.pl.scatter(raw,alpha=1,x="y_pixel",y="x_pixel",color="exp",title=meta_name,color_map=color_self,show=False,size=100000/raw.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/meta_gene.png", dpi=600)
plt.close()


**start**![](./sample_results/GFAP.png) **meta gene**![](./sample_results/meta_gene.png)

### 8. Multiple tissue sections analysis
In this section, we show an example on how to analysis multiple adjacent tissue sections using SpaGCN.

**Mouse brain anterior**![](./sample_results/MA1_lowres.png) **Mouse brain posterior**![](./sample_results/MP1_lowres.png)

#### 8.1 Read in data

In [35]:
adata1=sc.read("./data/Mouse_brain/MA1.h5ad")
adata2=sc.read("./data/Mouse_brain/MP1.h5ad")
img1=cv2.imread("./data/Mouse_brain/MA1_histology.tif")
img2=cv2.imread("./data/Mouse_brain/MP1_histology.tif")

#### 8.2 Extract color intensity

In [36]:
b=49
s=1
x_pixel1=adata1.obs["x4"].tolist()
y_pixel1=adata1.obs["x5"].tolist()
adata1.obs["color"]=spg.extract_color(x_pixel=x_pixel1, y_pixel=y_pixel1, image=img1, beta=b)
z_scale=np.max([np.std(x_pixel1), np.std(y_pixel1)])*s
adata1.obs["z"]=(adata1.obs["color"]-np.mean(adata1.obs["color"]))/np.std(adata1.obs["color"])*z_scale

x_pixel2=adata2.obs["x4"].tolist()
y_pixel2=adata2.obs["x5"].tolist()
adata2.obs["color"]=spg.extract_color(x_pixel=x_pixel2, y_pixel=y_pixel2, image=img2, beta=b)
z_scale=np.max([np.std(x_pixel2), np.std(y_pixel2)])*s
adata2.obs["z"]=(adata2.obs["color"]-np.mean(adata2.obs["color"]))/np.std(adata2.obs["color"])*z_scale
del img1, img2

#### 8.3 Modify coordinates to combine 2 sections

In [37]:
from anndata import AnnData
adata1.obs["x_pixel"]=x_pixel1
adata1.obs["y_pixel"]=y_pixel1
adata2.obs["x_pixel"]=x_pixel2-np.min(x_pixel2)+np.min(x_pixel1)
adata2.obs["y_pixel"]=y_pixel2-np.min(y_pixel2)+np.max(y_pixel1)
adata1.var_names_make_unique()
adata2.var_names_make_unique()
adata_all=AnnData.concatenate(adata1, adata2,join='inner',batch_key="dataset_batch",batch_categories=["0","1"])

#### 8.4 Integrate gene expression and histology into a Graph

In [38]:
X=np.array([adata_all.obs["x_pixel"], adata_all.obs["y_pixel"], adata_all.obs["z"]]).T.astype(np.float32)
adj=spg.pairwise_distance(X)

#### 8.5 Spatial domain detection using SpaGCN

In [39]:
sc.pp.normalize_per_cell(adata_all, min_counts=0)
sc.pp.log1p(adata_all)
p=0.5 
#Find the l value given p
l=spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

Run 1: l [0.01, 1000], p [0.0, 144.17116893743565]
Run 2: l [0.01, 500.005], p [0.0, 24.78992462158203]
Run 3: l [0.01, 250.0075], p [0.0, 3.649960994720459]
Run 4: l [125.00874999999999, 250.0075], p [0.4487175941467285, 3.649960994720459]
Run 5: l [125.00874999999999, 187.508125], p [0.4487175941467285, 1.5741894245147705]
Run 6: l [125.00874999999999, 156.2584375], p [0.4487175941467285, 0.9070142507553101]
Run 7: l [125.00874999999999, 140.63359375], p [0.4487175941467285, 0.6537595987319946]
Run 8: l [125.00874999999999, 132.821171875], p [0.4487175941467285, 0.5454769134521484]
recommended l =  128.91496093749998


In [40]:
res=1.0
seed=100
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
clf=spg.SpaGCN()
clf.set_l(l)
clf.train(adata_all,adj,init_spa=True,init="louvain",res=res, tol=5e-3, lr=0.05, max_epochs=200)
y_pred, prob=clf.predict()
adata_all.obs["pred"]= y_pred
adata_all.obs["pred"]=adata_all.obs["pred"].astype('category')

Initializing cluster centers with louvain, resolution =  1.0
Epoch  0
Epoch  10
Epoch  20
Epoch  30
delta_label  0.003141015043808894 < tol  0.005
Reach tolerance threshold. Stopping training.
Total epoch: 37


In [42]:
colors_use=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#bec1d4', '#bb7784', '#0000ff', '#111010', '#FFFF00',   '#1f77b4', '#800080', '#959595', 
 '#7d87b9', '#bec1d4', '#d6bcc0', '#bb7784', '#8e063b', '#4a6fe3', '#8595e1', '#b5bbe3', '#e6afb9', '#e07b91', '#d33f6a', '#11c638', '#8dd593', '#c6dec7', '#ead3c6', '#f0b98d', '#ef9708', '#0fcfc0', '#9cded6', '#d5eae7', '#f3e1eb', '#f6c4e1', '#f79cd4']
num_celltype=len(adata_all.obs["pred"].unique())
adata_all.uns["pred_colors"]=list(colors_use[:num_celltype])
ax=sc.pl.scatter(adata_all,alpha=1,x="y_pixel",y="x_pixel",color="pred",show=False,size=150000/adata_all.shape[0])
ax.set_aspect('equal', 'box')
ax.axes.invert_yaxis()
plt.savefig("./sample_results/mouse_barin_muti_sections_domains.png", dpi=600)
plt.close()

**SpaGCN mouse brain combines**![](./sample_results/mouse_barin_muti_sections_domains.png)

NameError: name 'adj1' is not defined